In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

In [3]:
dataset = pd.read_csv('datasets/Social_Network_Ads.csv')

In [4]:
dataset.shape

(400, 5)

In [5]:
dataset.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [6]:
dataset.tail()

,User ID,Gender,Age,EstimatedSalary,Purchased
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0
399,15594041,Female,49,36000,1


In [7]:
dataset.isnull().sum()

User ID            0
Gender             0
Age                0
EstimatedSalary    0
Purchased          0
dtype: int64

In [8]:
dataset.dtypes

User ID             int64
Gender             object
Age                 int64
EstimatedSalary     int64
Purchased           int64
dtype: object

In [9]:
dataset.describe()

,User ID,Age,EstimatedSalary,Purchased
count,4.000000e+02,400.000000,400.000000,400.000000
mean,1.569154e+07,37.655000,69742.500000,0.357500
std,7.165832e+04,10.482877,34096.960282,0.479864
min,1.556669e+07,18.000000,15000.000000,0.000000
25%,1.562676e+07,29.750000,43000.000000,0.000000
50%,1.569434e+07,37.000000,70000.000000,0.000000
75%,1.575036e+07,46.000000,88000.000000,1.000000
max,1.581524e+07,60.000000,150000.000000,1.000000


In [10]:
gender_mapping = {'Male': 1, 'Female': 0}
dataset['Gender'] = dataset['Gender'].map(gender_mapping)
X = dataset.drop(columns='Purchased', axis=1)
y = dataset['Purchased']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
classifier = LogisticRegression()
classifier.fit(X_train, y_train)

LogisticRegression()

In [13]:
y_pred = classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[57  1]
 [ 5 17]]


In [33]:
TP = cm[1, 1]
TN = cm[0, 0]
FP = cm[0, 1]
FN = cm[1, 0]

accuracy = (TP + TN) / (TP + TN + FP + FN)
error_rate = 1 - accuracy
precision = TP / (TP + FP)
recall = TP / (TP + FN)

print(f"Accuracy: {accuracy}\nError Rate: {error_rate} \nPrecision: {precision}\nRecall: {recall}")

Accuracy: 0.925
Error Rate: 0.07499999999999996 
Precision: 0.9444444444444444
Recall: 0.7727272727272727


***Logistic Regression without using sklearn***

In [34]:
# X_mean = np.mean(X, axis = 0)
# X_std = np.std(X, axis = 0)
# X = (X - X_mean) / X_std

In [46]:
# 1. Add intercept to X_train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

intercept_train = np.ones((X_train.shape[0], 1))
X_train = np.hstack((intercept_train, X_train))

# 2. Add intercept to X_test (This is the step that was missing!)
intercept_test = np.ones((X_test.shape[0], 1))
X_test = np.hstack((intercept_test, X_test))

In [47]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_loss(y, h):
    return (-y * np.log(h) - (1 - y) * np.log(1 - h)).mean()

def train(X, y, lr = 0.1, iterations = 1300):
    weights = np.zeros(X.shape[1])

    for i in range(iterations):
        z = np.dot(X, weights)
        h = sigmoid(z);

        gradient = np.dot(X.T, (h -  y)) / y.size
        weights -= lr * gradient

        if i % 100 == 0:
            loss = compute_loss(y, h)
            print(f"Iteration {i}: Loss {loss:.4f}")
            
    return weights

In [48]:
final_weights = train(X_train, y_train)

Iteration 0: Loss 0.6931
Iteration 100: Loss 0.4143
Iteration 200: Loss 0.3922
Iteration 300: Loss 0.3862
Iteration 400: Loss 0.3840
Iteration 500: Loss 0.3831
Iteration 600: Loss 0.3827
Iteration 700: Loss 0.3825
Iteration 800: Loss 0.3824
Iteration 900: Loss 0.3824
Iteration 1000: Loss 0.3823
Iteration 1100: Loss 0.3823
Iteration 1200: Loss 0.3823


In [51]:

probabilities = sigmoid(np.dot(X_test, final_weights))

# Convert probabilities to binary 0 or 1 predictions
y_pred = (probabilities >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)

TP = cm[1, 1]
TN = cm[0, 0]
FP = cm[0, 1]
FN = cm[1, 0]

accuracy = (TP + TN) / (TP + TN + FP + FN)
error_rate = 1 - accuracy
precision = TP / (TP + FP)
recall = TP / (TP + FN)

print(f"Accuracy: {accuracy}\nError Rate: {error_rate} \nPrecision: {precision}\nRecall: {recall}")

Accuracy: 0.9125
Error Rate: 0.08750000000000002 
Precision: 0.8947368421052632
Recall: 0.7727272727272727
